# DEMO 12: Metric Views — The Databricks Equivalent of DAX Measures

In Power BI, **DAX measures** let you define a calculation once (e.g. `Total Revenue = SUM(Sales[Amount])`) and reuse it across any visual with any combination of slicers. The measure evaluates dynamically based on the filter context.

**Metric Views** in Databricks achieve the same goal:
- Define measures (aggregations) and dimensions (grouping columns) **once** in Unity Catalog
- Query with **any combination** of dimensions at runtime
- Ensures **consistent metric definitions** across all dashboards and tools

| DAX Concept | Metric View Equivalent |
| --- | --- |
| DAX Measure | `measures:` in YAML definition |
| Dimension table fields | `dimensions:` in YAML definition |
| Filter context from slicers | `WHERE` clause in query |
| Implicit GROUP BY from visual | Explicit `GROUP BY ALL` in query |
| `CALCULATE` + filter | `FILTER (WHERE ...)` on a measure |

In this demo, you'll:
1. Create a metric view using the **Catalog Explorer UI** (no code required)
2. Query it with SQL in the **SQL Editor** using `MEASURE()` syntax

> **Note:** Run the setup cell below to create the source table, then follow the UI instructions to create the metric view in Catalog Explorer.

---

In [0]:
%run ./demo_12_setup

## Part 1: Creating a Metric View

A metric view has three core components:

1. **Source** — The table or query the metrics are calculated from
2. **Dimensions** — The columns you can GROUP BY (like rows/columns/slicers in a Power BI matrix)
3. **Measures** — The aggregation expressions (like DAX measures)

The definition uses **YAML** syntax inside a `CREATE VIEW ... WITH METRICS` statement.

**Key difference from standard views:**
- A standard view locks in the aggregation and grouping at creation time
- A metric view defines WHAT to aggregate but lets you choose HOW to group at query time

---

## Step-by-Step: Create the Metric View in Catalog Explorer

### Step 1: Navigate to your source table
1. Click **Catalog** in the left sidebar
2. Browse to your schema: `main` > `demo_metric_views_<your_username>` > `gold_orders`
3. Click the **gold_orders** table name to open the table details page

### Step 2: Create the metric view
1. Click the **Create** button (top right of the table details page)
2. Select **Metric view** from the dropdown
3. In the dialog:
   - **Name:** `order_metrics`
   - **Catalog:** `main`
   - **Schema:** `demo_metric_views_<your_username>`
4. Click **Create**

### Step 3: Define dimensions
The low-code UI editor opens. All source columns are added as dimensions automatically.

1. Review the auto-added dimensions (region, product_category, customer_segment, channel, order_date)
2. Optionally add a **calculated dimension** by clicking **+ Add** under Dimensions:
   - Expression: `DATE_TRUNC('month', order_date)`
   - Display name: `Order Month`
   - Comment: `Month the order was placed`

### Step 4: Define measures
Click **+ Add** under Measures to add each one:

| Measure Name | Expression | Comment |
| --- | --- | --- |
| Total Revenue | `SUM(revenue)` | Sum of all order revenue |
| Total Cost | `SUM(cost)` | Sum of all order costs |
| Order Count | `COUNT(order_id)` | Number of orders |
| Total Quantity | `SUM(quantity)` | Total units sold |
| Avg Order Value | `SUM(revenue) / COUNT(order_id)` | Average revenue per order |
| Profit Margin Pct | `(SUM(revenue) - SUM(cost)) / SUM(revenue) * 100` | Profit margin as percentage |

For each measure, you can optionally:
- Click **Preview** to see a sample result
- Add a **Format** (e.g., Currency USD for revenue)
- Add **Synonyms** for AI discoverability

### Step 5: Save
Click **Save** at the top of the editor.

Your metric view is now live in Unity Catalog and queryable from any SQL tool!

---

> **Alternative:** You can also switch to the **YAML** tab in the editor to see/edit the raw YAML definition directly. This is useful for version control or complex definitions.

### Reference: The SQL Equivalent (YAML Editor)

If you prefer code, the same metric view can be created with SQL. You can also view this by clicking the **YAML** tab in the Catalog Explorer editor:

```sql
CREATE OR REPLACE VIEW order_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: gold_orders
  dimensions:
    - name: Region
      expr: region
    - name: Product Category
      expr: product_category
    - name: Customer Segment
      expr: customer_segment
    - name: Channel
      expr: channel
    - name: Order Date
      expr: order_date
  measures:
    - name: Total Revenue
      expr: SUM(revenue)
    - name: Total Cost
      expr: SUM(cost)
    - name: Order Count
      expr: COUNT(order_id)
    - name: Total Quantity
      expr: SUM(quantity)
    - name: Avg Order Value
      expr: SUM(revenue) / COUNT(order_id)
    - name: Profit Margin Pct
      expr: (SUM(revenue) - SUM(cost)) / SUM(revenue) * 100
$$;
```

---

## Part 2: Querying a Metric View with MEASURE()

Querying a metric view is different from querying a regular table. The key rules:

| Rule | Example |
| --- | --- |
| Wrap measures with `MEASURE()` | `MEASURE(\`Total Revenue\`)` |
| Always use `GROUP BY ALL` | Required when selecting measures |
| Never use `SELECT *` | Must explicitly list dimensions and measures |
| Backtick-escape all names | `` \`Order Date\` ``, `` \`Total Revenue\` `` |

**Power BI Parallel:**
- In Power BI, you drag "Total Revenue" onto a visual and the engine applies the aggregation automatically
- In metric views, you write `MEASURE(\`Total Revenue\`)` and the engine applies the `SUM(revenue)` definition

Same concept, just explicit SQL syntax instead of drag-and-drop.

> **Now open the SQL Editor** (left sidebar) to run the queries below. Make sure to `USE SCHEMA` your demo schema first.

---

In [0]:
%sql
-- ============================================================
-- Query 1: Revenue by Region (simplest query)
-- ============================================================
-- Like dragging "Region" to rows and "Total Revenue" to values
-- in a Power BI matrix visual.

SELECT
  `Region`,
  MEASURE(`Total Revenue`) AS `Total Revenue`
FROM order_metrics
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- ============================================================
-- Query 2: Multiple measures by Category
-- ============================================================
-- Like adding multiple measures to the Values area of a matrix.

SELECT
  `Product Category`,
  MEASURE(`Total Revenue`) AS `Total Revenue`,
  MEASURE(`Order Count`) AS `Order Count`,
  MEASURE(`Avg Order Value`) AS `Avg Order Value`,
  MEASURE(`Profit Margin Pct`) AS `Profit Margin Pct`
FROM order_metrics
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- ============================================================
-- Query 3: Monthly revenue trend
-- ============================================================
-- Use date_trunc() to group by month (like a Power BI date hierarchy).
-- The same measures work at any time granularity!

SELECT
  date_trunc('MONTH', `Order Date`) AS `Month`,
  MEASURE(`Total Revenue`) AS `Total Revenue`,
  MEASURE(`Order Count`) AS `Order Count`
FROM order_metrics
GROUP BY ALL
ORDER BY `Month` ASC;

In [0]:
%sql
-- ============================================================
-- Query 4: Filter by dimension (like a Power BI slicer)
-- ============================================================
-- Same as selecting "Northeast" in a Region slicer.
-- The measure definitions don't change — just the filter context.

SELECT
  `Product Category`,
  `Channel`,
  MEASURE(`Total Revenue`) AS `Total Revenue`,
  MEASURE(`Profit Margin Pct`) AS `Profit Margin Pct`
FROM order_metrics
WHERE `Region` = 'Northeast'
  AND `Order Date` >= '2024-07-01'
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- ============================================================
-- Query 5: Filter by measure value (HAVING)
-- ============================================================
-- "Show me only segments where revenue exceeds $50,000"
-- In Power BI, this is like a visual-level filter on a measure.

SELECT
  `Customer Segment`,
  MEASURE(`Total Revenue`) AS `Total Revenue`,
  MEASURE(`Order Count`) AS `Order Count`
FROM order_metrics
GROUP BY ALL
HAVING `Total Revenue` > 50000
ORDER BY `Total Revenue` DESC;

## Part 3: Why This Matters — Define Once, Use Everywhere

The power of metric views is **consistency**:

| Without Metric Views | With Metric Views |
| --- | --- |
| Dashboard A calculates revenue as `SUM(amount)` | All dashboards use `MEASURE(\`Total Revenue\`)` |
| Dashboard B calculates revenue as `SUM(price * qty)` | Same definition everywhere |
| Numbers disagree → trust erodes | Numbers always match → single source of truth |

**Other benefits:**
- Stored in **Unity Catalog** — governed, versioned, discoverable
- Any team member can query the same metrics
- When the definition changes (e.g., exclude returns), it updates everywhere
- AI/BI dashboards and Genie can discover and use metric view definitions

**Power BI Parallel:** This is exactly what a Power BI dataset achieves — centralized measure definitions that all reports share. Metric views bring that same pattern to Databricks.

---

In [0]:
%sql
-- ============================================================
-- Query 6: Grand totals (no dimensions = overall summary)
-- ============================================================
-- Like a Power BI card visual showing a single KPI number.

SELECT
  MEASURE(`Total Revenue`) AS `Total Revenue`,
  MEASURE(`Total Cost`) AS `Total Cost`,
  MEASURE(`Order Count`) AS `Order Count`,
  MEASURE(`Avg Order Value`) AS `Avg Order Value`,
  MEASURE(`Profit Margin Pct`) AS `Profit Margin Pct`
FROM order_metrics
GROUP BY ALL;

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Metric View** | A Unity Catalog object that defines reusable measures + dimensions |
| **MEASURE()** | Required function wrapper to evaluate a defined measure |
| **GROUP BY ALL** | Always required when querying measures |
| **Dimensions** | Choose any combination at query time (like Power BI slicer/rows/columns) |
| **Consistency** | Same metric definition used across all dashboards — single source of truth |

**DAX → Metric View translation:**
- `Total Revenue = SUM(Sales[Revenue])` → `measures: [{name: Total Revenue, expr: SUM(revenue)}]`
- Drag field to visual → Add dimension to SELECT + GROUP BY ALL
- Slicer selection → WHERE clause
- Visual-level filter on measure → HAVING clause
- `CALCULATE` with filter → `FILTER (WHERE ...)` on a measure

**When to use metric views:**
- A metric is used across **multiple dashboards** (revenue, order count, conversion rate)
- You want a **single source of truth** for business KPIs
- Multiple teams need to query the **same metrics** with different groupings

---

**Next:** We'll put it all together in the challenge exercise, designing a complete dashboard-ready dataset.